In [2]:
import pandas as pd

df = pd.read_csv(
    "data/number_of_registered_users_of_fitbit_from_2014_to_2025_in_millions.csv"
)

df.columns = ["Year", "Registered_Users_Millions"]

print(df.to_string(index=False))

 Year  Registered_Users_Millions
 2014                         11
 2015                         29
 2016                         50
 2017                         75
 2018                         85
 2019                         90
 2020                         98
 2021                        111
 2022                        120
 2023                        128
 2024                        133
 2025                        137


In [3]:
df["New_Registered_Users_Millions"] = (
    df["Registered_Users_Millions"].diff()
)
print(df.to_string(index=False))

 Year  Registered_Users_Millions  New_Registered_Users_Millions
 2014                         11                            NaN
 2015                         29                           18.0
 2016                         50                           21.0
 2017                         75                           25.0
 2018                         85                           10.0
 2019                         90                            5.0
 2020                         98                            8.0
 2021                        111                           13.0
 2022                        120                            9.0
 2023                        128                            8.0
 2024                        133                            5.0
 2025                        137                            4.0


In [4]:
import numpy as np

years = df["Year"].to_numpy()
cumulative_users = df["Registered_Users_Millions"].to_numpy(dtype=float)

t = years - years[0]

print("First observation:", years[0])
print("Initial registered users (millions):", cumulative_users[0])
print("Number of observations:", len(t))


First observation: 2014
Initial registered users (millions): 11.0
Number of observations: 12


In [5]:
from scipy.optimize import least_squares

N0 = cumulative_users[0]

def bass_model(params, t):
    p, q, M = params
    f0 = N0 / M
    B = (p + q * f0) / (1 - f0)
    growth = np.exp(-(p + q) * t)

    return M * (B - p * growth) / (B + q * growth)

def residuals(params):
    return bass_model(params, t) - cumulative_users

result = least_squares(
    residuals,
    x0=[0.03, 0.30, 170],
    bounds=([0.000001, 0.000001, 137.000001],
            [1.0, 3.0, 1000.0]),
    max_nfev=20000
)

p, q, M = result.x

print("Optimization successful:", result.success)
print(f"p = {p:.6f}")
print(f"q = {q:.6f}")
print(f"M = {M:.3f} million registered users")
print(f"Sum of squared errors = {np.sum(result.fun**2):.4f}")


Optimization successful: True
p = 0.150388
q = 0.000001
M = 166.556 million registered users
Sum of squared errors = 131.3812


In [ ]:
# Evaluate the fitted Bass Model
predicted_users = bass_model(result.x, t)

sse = np.sum((cumulative_users - predicted_users) ** 2)
sst = np.sum((cumulative_users - np.mean(cumulative_users)) ** 2)

r_squared = 1 - sse / sst
rmse = np.sqrt(np.mean((cumulative_users - predicted_users) ** 2))

print(f"R-squared: {r_squared:.4f}")
print(f"RMSE: {rmse:.4f} million users")

comparison = pd.DataFrame({
    "Year": years,
    "Actual_Millions": cumulative_users,
    "Predicted_Millions": predicted_users
})

print(comparison.round(3).to_string(index=False))


R-squared: 0.9930
RMSE: 3.3088 million users
 Year  Actual_Millions  Predicted_Millions
 2014             11.0              11.000
 2015             29.0              32.720
 2016             50.0              51.407
 2017             75.0              67.484
 2018             85.0              81.317
 2019             90.0              93.219
 2020             98.0             103.459
 2021            111.0             112.269
 2022            120.0             119.848
 2023            128.0             126.370
 2024            133.0             131.981
 2025            137.0             136.808


In [ ]:
# STEPS 5, 7
p_oura = p
q_oura = q
oura_first_year_proxy = 2.3
def bass_fraction(t):
    exp_term = np.exp(-(p_oura + q_oura) * t)
    return (1 - exp_term) / (1 + (q_oura / p_oura) * exp_term)

M_oura = oura_first_year_proxy / bass_fraction(1)

periods = np.arange(1, 7)
years = np.arange(2025, 2031)

cumulative_adopters = M_oura * bass_fraction(periods)
previous_adopters = M_oura * bass_fraction(periods - 1)

annual_adopters = cumulative_adopters - previous_adopters

oura_forecast = pd.DataFrame({
    "Year": years,
    "Annual_Adopters_Millions": annual_adopters,
    "Cumulative_Adopters_Millions": cumulative_adopters
})

print("Oura Ring 4 - Bass Diffusion Model")
print(f"p = {p_oura:.6f}")
print(f"q = {q_oura:.6f}")
print(f"Calibrated M = {M_oura:.3f} million")
print()
print(oura_forecast.round(3).to_string(index=False))

Oura Ring 4 - Bass Diffusion Model
p = 0.150388
q = 0.000001
Calibrated M = 16.473 million

 Year  Annual_Adopters_Millions  Cumulative_Adopters_Millions
 2025                     2.300                         2.300
 2026                     1.979                         4.279
 2027                     1.703                         5.981
 2028                     1.465                         7.446
 2029                     1.260                         8.707
 2030                     1.084                         9.791


# HW1 — Bass Diffusion Model: Oura Ring 4

## Step 1 — Selected Innovation

The selected innovation is **Oura Ring 4**, featured in TIME's Best Inventions of 2025. Launched in October 2024, it is a smart ring that continuously monitors sleep, physical activity, heart rate, and other health indicators.

**Source:** [TIME — Oura Ring 4](https://time.com/collections/best-inventions-2025/7318554/oura-ring-4/)

## Step 2 — Similar Historical Innovation: Fitbit

Fitbit fitness trackers were selected as the historical comparison because both Fitbit and Oura Ring 4 use wearable sensors to monitor physical activity and health-related measurements. Both innovations provide personalized health insights through connected applications, making their functionality and technology comparable.

Fitbit played an important role in popularizing consumer health-monitoring wearables. Its historical adoption provides a useful reference for studying Oura Ring 4's potential diffusion. However, differences in product design, pricing, and target markets mean their adoption patterns may differ.

**Source:** [Fitbit — Historical Registered Users, Statista](https://www.statista.com/statistics/1327371/fitbit-registered-users/)

## Step 3 — Historical Data

Historical data were obtained from Statista's *Number of registered users of Fitbit worldwide from 2014 to 2025*. The dataset contains 12 annual observations, showing an increase from 11 million registered users in 2014 to 137 million in 2025.

Registered users are used as a proxy for cumulative adoption, although registrations do not necessarily represent unique first-time device purchasers.

**Source:** [Statista — Fitbit Registered Users, 2014–2025](https://www.statista.com/statistics/1327371/fitbit-registered-users/)

## Step 4 — Bass Model Parameter Estimation

Using Fitbit's historical data, the Bass Model parameters were estimated through nonlinear least squares in Python.

The estimated innovation coefficient is **p = 0.150388**, the imitation coefficient is **q = 0.000001**, and the estimated market potential is **M = 166.556 million**.

The model achieved an R-squared of **0.9930**, indicating a close fit to the observed cumulative registrations. However, the imitation coefficient reached its imposed lower bound, so the estimates should be interpreted cautiously.

## Step 5 — Oura Ring 4 Diffusion Prediction

Fitbit's estimated Bass parameters were applied to predict the diffusion of Oura Ring 4.

To support the forecast with relevant data, Oura's reported fiscal 2025 sales of **2.3 million rings** were used as an upper-bound proxy for first-year Ring 4 adoption. Calibrating the Bass Model to this assumption produced an estimated market potential of **16.473 million adopters**.

The model predicts cumulative adoption of approximately **4.279 million in 2026** and **9.791 million by 2030**.

These are conditional model estimates, not confirmed Ring 4 sales forecasts. Oura's reported sales include multiple ring generations and may include repeat purchases.

**Source:** [Oura Inc. — SEC Form S-1](https://www.sec.gov/Archives/edgar/data/2133022/000119312526381855/d119865ds1.htm)

## Step 6 — Geographic Scope

A **global market scope** was selected because the historical Fitbit dataset covers worldwide registered users, and Oura operates internationally.

Using a global scope maintains geographic consistency between the historical innovation and Oura Ring 4. Oura's international operations and market expansion further support this choice.

**Sources:** [Statista — Fitbit Registered Users](https://www.statista.com/statistics/1327371/fitbit-registered-users/) and [Oura Inc. — SEC Form S-1](https://www.sec.gov/Archives/edgar/data/2133022/000119312526381855/d119865ds1.htm)

## Step 7 — Estimated Adopters by Period

Annual adoption was estimated by calculating the difference between consecutive cumulative Bass Model predictions.

The estimated annual adopters are **2.300 million in 2025, 1.979 million in 2026, 1.703 million in 2027, 1.465 million in 2028, 1.260 million in 2029, and 1.084 million in 2030**.

The model predicts declining annual adoption alongside increasing cumulative adoption, reaching approximately **9.791 million cumulative adopters by 2030**.

These estimates are based on the Fitbit-derived Bass parameters and the Oura sales-based calibration assumption, rather than observed Ring 4 adoption counts.

---

## References

1. TIME. (2025). *Oura Ring 4 — The Best Inventions of 2025*. https://time.com/collections/best-inventions-2025/7318554/oura-ring-4/

2. Statista Research Department. (2026). *Number of registered users of Fitbit worldwide from 2014 to 2025 (in millions)*. https://www.statista.com/statistics/1327371/fitbit-registered-users/

3. Oura Inc. (2026). *Registration Statement on Form S-1*. U.S. Securities and Exchange Commission. https://www.sec.gov/Archives/edgar/data/2133022/000119312526381855/d119865ds1.htm